### Installation

In [ ]:
%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth
else:
    import torch; v = re.match(r"[0-9]{1,}\.[0-9]{1,}", str(torch.__version__)).group(0)
    xformers = "xformers==" + ("0.0.33.post1" if v=="2.9" else "0.0.32.post2" if v=="2.8" else "0.0.29.post3")
    !pip install --no-deps bitsandbytes accelerate {xformers} peft trl triton cut_cross_entropy unsloth_zoo
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth
!pip install transformers==4.56.2
!pip install --no-deps trl==0.22.2

### Unsloth

In [ ]:
# One must patch the DPO Trainer first!
from unsloth import PatchDPOTrainer

PatchDPOTrainer()

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [ ]:
from unsloth import FastLanguageModel
import torch
max_seq_length = 2048 # Choose any! We auto support RoPE Scaling internally!
dtype = None # None for auto detection. Float16 for Tesla T4, V100, Bfloat16 for Ampere+
load_in_4bit = True # Use 4bit quantization to reduce memory usage. Can be False.

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Llama-3.2-1B-Instruct-bnb-4bit", # Llama 3.2 1B Instruct (4-bit) for T4
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
    # token = "hf_...", # use one if using gated models like meta-llama/Llama-2-7b-hf
)

==((====))==  Unsloth 2026.2.1: Fast Llama patching. Transformers: 4.56.2.
   \\   /|    NVIDIA L4. Num GPUs = 1. Max memory: 22.034 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.0+cu128. CUDA: 8.9. CUDA Toolkit: 12.8. Triton: 3.5.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.33.post1. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/1.03G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/234 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/454 [00:00<?, ?B/s]

In [ ]:
# @title Alignment Handbook utils
import os
import re
from typing import List, Literal, Optional

from datasets import DatasetDict, concatenate_datasets, load_dataset, load_from_disk
from datasets.builder import DatasetGenerationError


DEFAULT_CHAT_TEMPLATE = "{% for message in messages %}\n{% if message['role'] == 'user' %}\n{{ '<|user|>\n' + message['content'] + eos_token }}\n{% elif message['role'] == 'system' %}\n{{ '<|system|>\n' + message['content'] + eos_token }}\n{% elif message['role'] == 'assistant' %}\n{{ '<|assistant|>\n'  + message['content'] + eos_token }}\n{% endif %}\n{% if loop.last and add_generation_prompt %}\n{{ '<|assistant|>' }}\n{% endif %}\n{% endfor %}"


def apply_chat_template(
    example,
    tokenizer,
    task: Literal["sft", "generation", "rm", "dpo"] = "sft",
    assistant_prefix="<|assistant|>\n",
):
    def _strip_prefix(s, pattern):
        # Use re.escape to escape any special characters in the pattern
        return re.sub(f"^{re.escape(pattern)}", "", s)

    if task in ["sft", "generation"]:
        messages = example["messages"]
        # We add an empty system message if there is none
        if messages[0]["role"] != "system":
            messages.insert(0, {"role": "system", "content": ""})
        example["text"] = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True if task == "generation" else False,
        )
    elif task == "rm":
        if all(k in example.keys() for k in ("chosen", "rejected")):
            chosen_messages = example["chosen"]
            rejected_messages = example["rejected"]
            # We add an empty system message if there is none
            if chosen_messages[0]["role"] != "system":
                chosen_messages.insert(0, {"role": "system", "content": ""})
            if rejected_messages[0]["role"] != "system":
                rejected_messages.insert(0, {"role": "system", "content": ""})
            example["text_chosen"] = tokenizer.apply_chat_template(
                chosen_messages, tokenize=False
            )
            example["text_rejected"] = tokenizer.apply_chat_template(
                rejected_messages, tokenize=False
            )
        else:
            raise ValueError(
                f"Could not format example as dialogue for `rm` task! Require `[chosen, rejected]` keys but found {list(example.keys())}"
            )
    elif task == "dpo":
        if all(k in example.keys() for k in ("chosen", "rejected")):
            # Compared to reward modeling, we filter out the prompt, so the text is everything after the last assistant token
            prompt_messages = [
                [msg for msg in example["chosen"] if msg["role"] == "user"][0]
            ]
            # Insert system message
            if example["chosen"][0]["role"] != "system":
                prompt_messages.insert(0, {"role": "system", "content": ""})
            else:
                prompt_messages.insert(0, example["chosen"][0])
            # TODO: handle case where chosen/rejected also have system messages
            chosen_messages = example["chosen"][1:]
            rejected_messages = example["rejected"][1:]
            example["text_chosen"] = tokenizer.apply_chat_template(
                chosen_messages, tokenize=False
            )
            example["text_rejected"] = tokenizer.apply_chat_template(
                rejected_messages, tokenize=False
            )
            example["text_prompt"] = tokenizer.apply_chat_template(
                prompt_messages, tokenize=False, add_generation_prompt=True
            )
            example["text_chosen"] = _strip_prefix(
                example["text_chosen"], assistant_prefix
            )
            example["text_rejected"] = _strip_prefix(
                example["text_rejected"], assistant_prefix
            )
        else:
            raise ValueError(
                f"Could not format example as dialogue for `dpo` task! Require `[chosen, rejected]` keys but found {list(example.keys())}"
            )
    else:
        raise ValueError(
            f"Task {task} not supported, please ensure that the provided task is one of {['sft', 'generation', 'rm', 'dpo']}"
        )
    return example


def get_datasets(
    data_config: dict,
    splits: List[str] = ["train", "test"],
    shuffle: bool = True,
) -> DatasetDict:
    """
    Loads one or more datasets with varying training set proportions.

    Args:
        data_config (`DataArguments` or `dict`):
            Dataset configuration and split proportions.
        splits (`List[str]`, *optional*, defaults to `['train', 'test']`):
            Dataset splits to load and mix. Assumes the splits exist in all datasets and have a `train_` or `test_` prefix.
        shuffle (`bool`, *optional*, defaults to `True`):
            Whether to shuffle the training and testing/validation data.

    Returns
        [`DatasetDict`]: The dataset dictionary containing the loaded datasets.
    """

    if type(data_config) is dict:
        # Structure of the input is:
        #     dataset_mixer = {
        #             "dataset1": 0.5,
        #             "dataset1": 0.3,
        #             "dataset1": 0.2,
        #         }
        dataset_mixer = data_config
    else:
        raise ValueError(f"Data config {data_config} not recognized.")

    raw_datasets = mix_datasets(dataset_mixer, splits=splits, shuffle=shuffle)
    return raw_datasets


def mix_datasets(
    dataset_mixer: dict, splits: Optional[List[str]] = None, shuffle=True
) -> DatasetDict:
    """
    Loads and mixes datasets according to proportions specified in `dataset_mixer`.

    Args:
        dataset_mixer (`dict`):
            Dictionary containing the dataset names and their training proportions. By default, all test proportions are 1.
        splits (Optional[List[str]], *optional*, defaults to `None`):
            Dataset splits to load and mix. Assumes the splits exist in all datasets and have a `train_` or `test_` prefix.
        shuffle (`bool`, *optional*, defaults to `True`):
            Whether to shuffle the training and testing/validation data.
    """
    raw_datasets = DatasetDict()
    raw_train_datasets = []
    raw_val_datasets = []
    fracs = []
    for ds, frac in dataset_mixer.items():
        fracs.append(frac)
        for split in splits:
            try:
                # Try first if dataset on a Hub repo
                dataset = load_dataset(ds, split=split)
            except DatasetGenerationError:
                # If not, check local dataset
                dataset = load_from_disk(os.path.join(ds, split))

            if "train" in split:
                raw_train_datasets.append(dataset)
            elif "test" in split:
                raw_val_datasets.append(dataset)
            else:
                raise ValueError(
                    f"Split type {split} not recognized as one of test or train."
                )

    if any(frac < 0 for frac in fracs):
        raise ValueError("Dataset fractions cannot be negative.")

    if len(raw_train_datasets) > 0:
        train_subsets = []
        for dataset, frac in zip(raw_train_datasets, fracs):
            train_subset = dataset.select(range(int(frac * len(dataset))))
            train_subsets.append(train_subset)
        if shuffle:
            raw_datasets["train"] = concatenate_datasets(train_subsets).shuffle(seed=42)
        else:
            raw_datasets["train"] = concatenate_datasets(train_subsets)
    # No subsampling for test datasets to enable fair comparison across models
    if len(raw_val_datasets) > 0:
        if shuffle:
            raw_datasets["test"] = concatenate_datasets(raw_val_datasets).shuffle(
                seed=42
            )
        else:
            raw_datasets["test"] = concatenate_datasets(raw_val_datasets)

    if len(raw_datasets) == 0:
        raise ValueError(
            f"Dataset {dataset_mixer} not recognized with split {split}. Check the dataset has been correctly formatted."
        )

    return raw_datasets

<a name="Data"></a>
### Data Prep


In [ ]:
from pathlib import Path
import pandas as pd

files = list(Path("/content/RamsayData/Train").glob("*.csv")) + list(Path("/content/RamsayData/Train").glob("*.CSV"))

for f in files:
    try:
        pd.read_csv(f, encoding="cp1252")  # using your earlier encoding issue
    except Exception as e:
        print("FAILED:", f.name)
        raise


In [ ]:
from pathlib import Path
import pandas as pd
from datasets import Dataset, DatasetDict

BASE = Path("/content/drive/MyDrive/RamsayData")

TRAIN_N = 500
TEST_N  = 100
SEED    = 42

def read_csv_safe(path: Path) -> pd.DataFrame:
    # sep=None + engine="python" lets pandas sniff the delimiter (comma/semicolon/etc.) :contentReference[oaicite:1]{index=1}
    for enc in ("utf-8", "utf-8-sig", "cp1252", "latin1"):
        try:
            return pd.read_csv(
                path,
                encoding=enc,
                sep=None,
                engine="python",
                on_bad_lines="skip",
            )
        except UnicodeDecodeError:
            pass

    # last resort: replace bad characters instead of crashing :contentReference[oaicite:2]{index=2}
    return pd.read_csv(
        path,
        encoding="utf-8",
        encoding_errors="replace",
        sep=None,
        engine="python",
        on_bad_lines="skip",
    )

def load_split(split_name: str) -> pd.DataFrame:
    folder = BASE / split_name
    if not folder.exists():
        raise FileNotFoundError(f"Folder not found: {folder}")

    # rglob = includes CSVs in subfolders too
    files = sorted(list(folder.rglob("*.csv")) + list(folder.rglob("*.CSV")))
    if not files:
        raise FileNotFoundError(f"No CSVs found in: {folder}")

    print(f"[{split_name}] files found:", len(files))
    # Uncomment if you want to see names:
    # print("\n".join(f" - {f.name}" for f in files))

    df = pd.concat((read_csv_safe(f) for f in files), ignore_index=True)

    # Normalize columns
    df.columns = [c.strip().lower() for c in df.columns]

    # If normalization created duplicates, keep the first copy
    if df.columns.duplicated().any():
        dupes = df.columns[df.columns.duplicated()].tolist()
        print(f"[{split_name}] Duplicate columns after normalization (keeping first):", dupes)
        df = df.loc[:, ~df.columns.duplicated(keep="first")]

    # Require the 3 columns (case-insensitive already)
    required = {"question", "ramsay", "polite"}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f"[{split_name}] Missing columns {missing}. Found: {list(df.columns)}")

    # Map to DPO schema
    df = df.rename(columns={
        "question": "prompt",
        "ramsay": "chosen",
        "polite": "rejected",
    })[["prompt", "chosen", "rejected"]].dropna()

    # Cleanup
    for c in ["prompt", "chosen", "rejected"]:
        df[c] = df[c].astype(str).str.strip()
    df = df[(df["prompt"] != "") & (df["chosen"] != "") & (df["rejected"] != "")]

    print(f"[{split_name}] valid rows after cleanup:", len(df))
    return df.reset_index(drop=True)

# Load ONCE
train_df = load_split("Train")
test_df  = load_split("Test")

# Enforce required sample sizes
if len(train_df) < TRAIN_N:
    raise ValueError(f"Train has only {len(train_df)} valid rows (<{TRAIN_N}). "
                     f"Add more CSV data or fix skipped/bad rows.")
train_df = train_df.sample(n=TRAIN_N, random_state=SEED, replace=False).reset_index(drop=True)

if len(test_df) < TEST_N:
    print(f"Warning: Test has only {len(test_df)} valid rows (<{TEST_N}). Using all of them.")
else:
    test_df = test_df.sample(n=TEST_N, random_state=SEED, replace=False).reset_index(drop=True)

raw_datasets = DatasetDict({
    "train": Dataset.from_pandas(train_df, preserve_index=False),
    "test":  Dataset.from_pandas(test_df,  preserve_index=False),
})

print("Final Train size:", len(raw_datasets["train"]))
print("Final Test size:", len(raw_datasets["test"]))
raw_datasets


[Train] files found: 11
[Train] Duplicate columns after normalization (keeping first): ['question']
[Train] valid rows after cleanup: 601
[Test] files found: 1
[Test] valid rows after cleanup: 100
Final Train size: 500
Final Test size: 100


DatasetDict({
    train: Dataset({
        features: ['prompt', 'chosen', 'rejected'],
        num_rows: 500
    })
    test: Dataset({
        features: ['prompt', 'chosen', 'rejected'],
        num_rows: 100
    })
})

We shall print a random item from the dataset

In [ ]:
import pprint

row = raw_datasets['train'][0]
pprint.pprint(row['prompt'])
pprint.pprint(row['chosen'])
pprint.pprint(row['rejected'])


'What is weight initialization and why does it matter?'
('It’s the model’s first step—start it wrong and the whole training goes off '
 'the rails like serving raw chicken on opening night!')
('Weight initialization sets the starting values for a model’s parameters, '
 'helping ensure stable and effective training.')


In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 32,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 32,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
    use_rslora = False,
    loftq_config = None,
)

Unsloth 2026.2.1 patched 16 layers with 16 QKV layers, 16 O layers and 16 MLP layers.


<a name="Train"></a>
### Train the DPO model
Now let's train our model. We do 3 epochs on 0.5% of the dataset to speed things up.

In [ ]:
# One must patch the DPO Trainer first!
from unsloth import PatchDPOTrainer

PatchDPOTrainer()

In [ ]:
from transformers import TrainingArguments
from trl import DPOTrainer, DPOConfig
dpo_trainer = DPOTrainer(
    model = model,
    ref_model = None,
    args = DPOConfig(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_ratio = 0.1,
        num_train_epochs = 3,
        learning_rate = 5e-6,
        logging_steps = 1,
        eval_strategy = "epoch",
        per_device_eval_batch_size = 1,
        save_strategy = "no",
        optim = "adamw_8bit",
        weight_decay = 0.0,
        lr_scheduler_type = "linear",
        seed = 42,
        output_dir = "outputs",
        report_to = "none",
    ),
    beta = 0.1,
    train_dataset = raw_datasets["train"],
    eval_dataset = raw_datasets["test"],
    tokenizer = tokenizer,
    max_length = 1024,
    max_prompt_length = 512,
)

Extracting prompt in train dataset (num_proc=16):   0%|          | 0/500 [00:00<?, ? examples/s]

Applying chat template to train dataset (num_proc=16):   0%|          | 0/500 [00:00<?, ? examples/s]

Tokenizing train dataset (num_proc=16):   0%|          | 0/500 [00:00<?, ? examples/s]

Extracting prompt in eval dataset (num_proc=16):   0%|          | 0/100 [00:00<?, ? examples/s]

Applying chat template to eval dataset (num_proc=16):   0%|          | 0/100 [00:00<?, ? examples/s]

Tokenizing eval dataset (num_proc=16):   0%|          | 0/100 [00:00<?, ? examples/s]

In [ ]:
dpo_trainer.train()


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 500 | Num Epochs = 3 | Total steps = 189
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 22,544,384 of 1,258,358,784 (1.79% trained)


Epoch,Training Loss,Validation Loss,rewards / chosen,rewards / rejected,rewards / accuracies,rewards / margins,logps / chosen,logps / rejected,logits / chosen,logits / rejected,eval_logits / chosen,eval_logits / rejected,nll_loss
1,0.515100,0.521456,0.314090,-0.071216,1.000000,0.385306,-140.301239,-67.313271,1.216561,1.291975,0,0,0
2,0.266000,0.310208,0.802323,-0.258790,1.000000,1.061113,-135.418915,-69.189011,1.234412,1.404493,No Log,No Log,No Log
3,0.195800,0.255566,0.952944,-0.359764,1.000000,1.312708,-133.912720,-70.198753,1.235834,1.439629,No Log,No Log,No Log


TrainOutput(global_step=189, training_loss=0.3945264980275795, metrics={'train_runtime': 266.0184, 'train_samples_per_second': 5.639, 'train_steps_per_second': 0.71, 'total_flos': 0.0, 'train_loss': 0.3945264980275795, 'epoch': 3.0})

In [ ]:
SAVE_DIR = "/content/drive/MyDrive/RamsayData/Outcome/dpo_ramsay_llama32_1b_final"

model = dpo_trainer.model

model.save_pretrained_merged(SAVE_DIR, tokenizer, save_method="merged_16bit")

print("Saved merged 16-bit model to:", SAVE_DIR)


config.json:   0%|          | 0.00/894 [00:00<?, ?B/s]

Found HuggingFace hub cache directory: /root/.cache/huggingface/hub
Checking cache directory for required files...
Cache check failed: model.safetensors not found in local cache.
Not all required files found in cache. Will proceed with downloading.
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


Unsloth: Preparing safetensor model files: 100%|██████████| 1/1 [00:00<00:00, 2153.13it/s]


Note: tokenizer.model not found (this is OK for non-SentencePiece models)


Unsloth: Merging weights into 16bit: 100%|██████████| 1/1 [00:40<00:00, 40.62s/it]


Unsloth: Merge process complete. Saved to `/content/drive/MyDrive/RamsayData/Outcome/dpo_ramsay_llama32_1b_final`
Saved merged 16-bit model to: /content/drive/MyDrive/RamsayData/Outcome/dpo_ramsay_llama32_1b_final


In [ ]:
import pandas as pd
import numpy as np
import torch

FastLanguageModel.for_inference(model)
DEVICE = "cuda"

IN_CSV  = "/content/drive/MyDrive/RamsayData/Test/ramsay_style_0112(Sheet1).csv"
OUT_CSV = "/content/drive/MyDrive/RamsayData/Outcome/ramsay_task5_with_results.csv"

def read_csv_safe(path):
    for enc in ("utf-8", "utf-8-sig", "cp1252", "latin1"):
        try:
            return pd.read_csv(path, encoding=enc, sep=None, engine="python", on_bad_lines="skip")
        except UnicodeDecodeError:
            pass
    return pd.read_csv(path, encoding="utf-8", encoding_errors="replace",
                       sep=None, engine="python", on_bad_lines="skip")

df = read_csv_safe(IN_CSV)

def format_prompt(q: str) -> str:
    return f"You are Gordon Ramsay.\nUser: {q}\nAssistant:"

def generate_answer(question, max_new_tokens=128):
    prompt = format_prompt(str(question))
    inputs = tokenizer(
      prompt,
      return_tensors="pt",
      truncation=True,
      max_length=512
    ).to(DEVICE)
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=256,
            do_sample=True, # Changed to True for sampling
            temperature=0.7,
            top_p=0.9,
            num_beams=1,                 # Keep num_beams=1 for sampling, set to >1 for beam search
            stop_strings=["\nUser:", "\nUSER:"],
            tokenizer=tokenizer,
        )

    gen_ids = out[0, inputs["input_ids"].shape[1]:]
    return tokenizer.decode(gen_ids, skip_special_tokens=True).strip()

# --- Generate outputs ---
print("Generating model outputs...")
df["model_output"] = df["Question"].astype(str).apply(generate_answer)

df.to_csv(OUT_CSV, index=False)
print("Saved:", OUT_CSV)
df.head()

Generating model outputs...
Saved: /content/drive/MyDrive/RamsayData/Outcome/ramsay_task5_with_results.csv


,﻿ID,Question,Polite,Ramsay,model_output
0,1,I used 90% dropout on the hidden layer.,That is too high; the model can't learn. Try 0...,90% DROPOUT? You’ve thrown away 90% of the bra...,What the bloody hell are you talking about? Yo...
1,2,Why is my loss not decreasing?,Your learning rate might be too high. Try lowe...,IT'S NOT MOVING! It's stuck! Just like your br...,"(in a thick British accent) Oh, for crying out..."
2,3,I trained on the test data.,That causes data leakage. You must never train...,YOU DONKEY! You’re marking your own homework! ...,"(sigh) Oh, bloody hell! What's the bloody prob..."
3,4,My accuracy is 99% on an imbalanced dataset.,"Accuracy is misleading here. Check Precision, ...",99% accuracy on a 1% target? My cat could gues...,"(pausing, grumbling) Oh, for crying out loud, ..."
4,5,I didn't normalize my inputs.,Normalization helps gradients converge faster ...,You’ve got data ranges all over the place! It’...,"(scoffing) Oh, for crying out loud, boy! You t..."


In [ ]:
# google_electra-base from Step 2

import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

device = "cuda" if torch.cuda.is_available() else "cpu"

MODEL_DIR = "/content/drive/MyDrive/RamsayData/Outcome/google_electra-base-discriminator/checkpoint-1080" # Changed to specific checkpoint

tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR, fix_mistral_regex=True, local_files_only=True)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_DIR, num_labels=1, local_files_only=True).to(device)
model.eval()

def predict_similarity(s1, s2, clip_0_5=True):
    inputs = tokenizer(s1, s2, return_tensors="pt", truncation=True, padding=True).to(device)
    with torch.no_grad():
        outputs = model(**inputs)
    score = outputs.logits.squeeze().item()
    if clip_0_5:
        score = max(0.0, min(5.0, score))

    # Round to nearest integer to get 0-5 without decimals
    return int(round(score))

# model_output vs Ramsay
df["electra_sim_out_vs_ramsay"] = [
    predict_similarity(o, r) for o, r in zip(df["model_output"].astype(str), df["Ramsay"].astype(str))
]

print("Avg similarity(output, Ramsay):", df["electra_sim_out_vs_ramsay"].mean())

OUT_SIM = "/content/drive/MyDrive/RamsayData/Outcome/ramsay_step8_electra_similarity.csv"
df.to_csv(OUT_SIM, index=False)
print("Saved:", OUT_SIM)
df[["Question","model_output","electra_sim_out_vs_ramsay"]].head()

Avg similarity(output, Ramsay): 2.02
Saved: /content/drive/MyDrive/RamsayData/Outcome/ramsay_step8_electra_similarity.csv


,Question,model_output,electra_sim_out_vs_ramsay
0,I used 90% dropout on the hidden layer.,What the bloody hell are you talking about? Yo...,3
1,Why is my loss not decreasing?,"(in a thick British accent) Oh, for crying out...",2
2,I trained on the test data.,"(sigh) Oh, bloody hell! What's the bloody prob...",2
3,My accuracy is 99% on an imbalanced dataset.,"(pausing, grumbling) Oh, for crying out loud, ...",3
4,I didn't normalize my inputs.,"(scoffing) Oh, for crying out loud, boy! You t...",2


In [ ]:
# JUDGE 1: Llama 3.1 8B Instruct
import torch
import gc
from tqdm.auto import tqdm
from unsloth import FastLanguageModel
import re
import pandas as pd

# 1. Clean Memory
if 'model' in globals(): del model
if 'tokenizer' in globals(): del tokenizer
torch.cuda.empty_cache()
gc.collect()

JUDGE_MODEL = "unsloth/Llama-3.1-8B-Instruct-bnb-4bit"
print(f"Loading Judge: {JUDGE_MODEL}...")

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = JUDGE_MODEL,
    max_seq_length = 2048,
    dtype = None,
    load_in_4bit = True,
)
FastLanguageModel.for_inference(model)
device = "cuda" if torch.cuda.is_available() else "cpu"

SYSTEM = (
    "You are a strict semantic similarity judge.\n"
    "Score how close two sentences are in meaning, ignoring style and tone.\n"
    "Scale:\n"
    "5 = same meaning\n"
    "4 = mostly same meaning\n"
    "3 = related but missing/changed key info\n"
    "2 = weakly related\n"
    "1 = almost unrelated\n"
    "0 = completely unrelated\n\n"
    "IMPORTANT:\n"
    "- Output EXACTLY in this format: Score: <number> | Reason: <short explanation>"
)

def llm_semantic_score_with_reason(pred_sentence: str, gold_sentence: str):
    user_msg = (f"Sentence A: {pred_sentence}\nSentence B: {gold_sentence}\nCompare meaning.")

    messages = [{"role": "system", "content": SYSTEM}, {"role": "user", "content": user_msg}]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=128, do_sample=False, temperature=0.0)
    text = tokenizer.decode(out[0], skip_special_tokens=True)

    match = re.search(r"Score:\s*([0-5](?:\.\d+)?)\s*\|\s*Reason:\s*(.+)", text)
    if match:
        return float(match.group(1)), match.group(2).strip()
    m_score = re.search(r"Score:\s*([0-5])", text)
    if m_score:
        return float(m_score.group(1)), text.replace(m_score.group(0), "").strip()
    return 0.0, text.strip()

# Run Judge
scores, reasons = [], []
print("Running Llama 3.1 8B Judge...")
for pred, gold in tqdm(zip(df["model_output"].astype(str), df["Ramsay"].astype(str)), total=len(df)):
    s, r = llm_semantic_score_with_reason(pred, gold)
    scores.append(s)
    reasons.append(r)

# Save Results
df["llm_sim_llama3_8b"] = scores
df["llm_reason_llama3_8b"] = reasons

print("Avg Llama 3.1 8B Similarity:", df["llm_sim_llama3_8b"].mean())
OUT_CSV = "/content/drive/MyDrive/RamsayData/Outcome/ramsay_judge_llama3_8b.csv"
df.to_csv(OUT_CSV, index=False)
print("Saved:", OUT_CSV)

Loading Judge: unsloth/Llama-3.1-8B-Instruct-bnb-4bit...
==((====))==  Unsloth 2026.2.1: Fast Llama patching. Transformers: 4.56.2.
   \\   /|    NVIDIA L4. Num GPUs = 1. Max memory: 22.034 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.0+cu128. CUDA: 8.9. CUDA Toolkit: 12.8. Triton: 3.5.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.33.post1. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/5.70G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/454 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

Running Llama 3.1 8B Judge...


  0%|          | 0/100 [00:00<?, ?it/s]

Avg Llama 3.1 8B Similarity: 3.23
Saved: /content/drive/MyDrive/RamsayData/Outcome/ramsay_judge_llama3_8b.csv


In [ ]:
# JUDGE 2: Qwen 2.5 1.5B Instruct
import torch
import gc
from tqdm.auto import tqdm
from unsloth import FastLanguageModel
import re

# 1. Clean Memory
if 'model' in globals(): del model
if 'tokenizer' in globals(): del tokenizer
torch.cuda.empty_cache()
gc.collect()

JUDGE_MODEL = "unsloth/Qwen2.5-1.5B-Instruct-bnb-4bit"
print(f"Loading Judge: {JUDGE_MODEL}...")

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = JUDGE_MODEL,
    max_seq_length = 2048,
    dtype = None,
    load_in_4bit = True,
)
FastLanguageModel.for_inference(model)
device = "cuda" if torch.cuda.is_available() else "cpu"

SYSTEM = (
    "You are a strict semantic similarity judge.\n"
    "Score how close two sentences are in meaning, ignoring style and tone.\n"
    "Scale:\n"
    "5 = same meaning\n"
    "4 = mostly same meaning\n"
    "3 = related but missing/changed key info\n"
    "2 = weakly related\n"
    "1 = almost unrelated\n"
    "0 = completely unrelated\n\n"
    "IMPORTANT:\n"
    "- Output EXACTLY in this format: Score: <number> | Reason: <short explanation>"
)

def llm_semantic_score_with_reason(pred_sentence: str, gold_sentence: str):
    user_msg = (f"Sentence A: {pred_sentence}\nSentence B: {gold_sentence}\nCompare meaning.")
    messages = [{"role": "system", "content": SYSTEM}, {"role": "user", "content": user_msg}]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=128, do_sample=False, temperature=0.0)
    text = tokenizer.decode(out[0], skip_special_tokens=True)
    match = re.search(r"Score:\s*([0-5](?:\.\d+)?)\s*\|\s*Reason:\s*(.+)", text)
    if match:
        return float(match.group(1)), match.group(2).strip()
    m_score = re.search(r"Score:\s*([0-5])", text)
    if m_score:
        return float(m_score.group(1)), text.replace(m_score.group(0), "").strip()
    return 0.0, text.strip()

# Run Judge
scores, reasons = [], []
print("Running Qwen 2.5 1.5B Judge...")
for pred, gold in tqdm(zip(df["model_output"].astype(str), df["Ramsay"].astype(str)), total=len(df)):
    s, r = llm_semantic_score_with_reason(pred, gold)
    scores.append(s)
    reasons.append(r)

# Save Results
df["llm_sim_qwen25_1_5b"] = scores
df["llm_reason_qwen25_1_5b"] = reasons

print("Avg Qwen 2.5 1.5B Similarity:", df["llm_sim_qwen25_1_5b"].mean())
OUT_CSV = "/content/drive/MyDrive/RamsayData/Outcome/ramsay_judge_qwen25_1_5b.csv"
df.to_csv(OUT_CSV, index=False)
print("Saved:", OUT_CSV)

Loading Judge: unsloth/Qwen2.5-1.5B-Instruct-bnb-4bit...
==((====))==  Unsloth 2026.2.1: Fast Qwen2 patching. Transformers: 4.56.2.
   \\   /|    NVIDIA L4. Num GPUs = 1. Max memory: 22.034 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.0+cu128. CUDA: 8.9. CUDA Toolkit: 12.8. Triton: 3.5.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.33.post1. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/1.14G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/270 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

Running Qwen 2.5 1.5B Judge...


  0%|          | 0/100 [00:00<?, ?it/s]

Avg Qwen 2.5 1.5B Similarity: 5.0
Saved: /content/drive/MyDrive/RamsayData/Outcome/ramsay_judge_qwen25_1_5b.csv


In [ ]:
# JUDGE 3: Qwen 2.5 7B Instruct
import torch
import gc
from tqdm.auto import tqdm
from unsloth import FastLanguageModel
import re

# 1. Clean Memory
if 'model' in globals(): del model
if 'tokenizer' in globals(): del tokenizer
torch.cuda.empty_cache()
gc.collect()

JUDGE_MODEL = "unsloth/Qwen2.5-7B-Instruct-bnb-4bit"
print(f"Loading Judge: {JUDGE_MODEL}...")

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = JUDGE_MODEL,
    max_seq_length = 2048,
    dtype = None,
    load_in_4bit = True,
)
FastLanguageModel.for_inference(model)
device = "cuda" if torch.cuda.is_available() else "cpu"

SYSTEM = (
    "You are a strict semantic similarity judge.\n"
    "Score how close two sentences are in meaning, ignoring style and tone.\n"
    "Scale:\n"
    "5 = same meaning\n"
    "4 = mostly same meaning\n"
    "3 = related but missing/changed key info\n"
    "2 = weakly related\n"
    "1 = almost unrelated\n"
    "0 = completely unrelated\n\n"
    "IMPORTANT:\n"
    "- Output EXACTLY in this format: Score: <number> | Reason: <short explanation>"
)

def llm_semantic_score_with_reason(pred_sentence: str, gold_sentence: str):
    user_msg = (f"Sentence A: {pred_sentence}\nSentence B: {gold_sentence}\nCompare meaning.")
    messages = [{"role": "system", "content": SYSTEM}, {"role": "user", "content": user_msg}]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=128, do_sample=False, temperature=0.0)
    text = tokenizer.decode(out[0], skip_special_tokens=True)
    match = re.search(r"Score:\s*([0-5](?:\.\d+)?)\s*\|\s*Reason:\s*(.+)", text)
    if match:
        return float(match.group(1)), match.group(2).strip()
    m_score = re.search(r"Score:\s*([0-5])", text)
    if m_score:
        return float(m_score.group(1)), text.replace(m_score.group(0), "").strip()
    return 0.0, text.strip()

# Run Judge
scores, reasons = [], []
print("Running Qwen 2.5 7B Judge...")
for pred, gold in tqdm(zip(df["model_output"].astype(str), df["Ramsay"].astype(str)), total=len(df)):
    s, r = llm_semantic_score_with_reason(pred, gold)
    scores.append(s)
    reasons.append(r)

# Save Results
df["llm_sim_qwen25_7b"] = scores
df["llm_reason_qwen25_7b"] = reasons

print("Avg Qwen 2.5 7B Similarity:", df["llm_sim_qwen25_7b"].mean())
OUT_CSV = "/content/drive/MyDrive/RamsayData/Outcome/ramsay_judge_qwen25_7b.csv"
df.to_csv(OUT_CSV, index=False)
print("Saved:", OUT_CSV)

# Final Comparison of all 3
print("\n--- Final Comparison of All Judges ---")
print(f"Llama 3.1 8B Avg:   {df['llm_sim_llama3_8b'].mean():.2f}")
print(f"Qwen 2.5 1.5B Avg:  {df['llm_sim_qwen25_1_5b'].mean():.2f}")
print(f"Qwen 2.5 7B Avg:    {df['llm_sim_qwen25_7b'].mean():.2f}")
if 'electra_sim_out_vs_ramsay' in df.columns:
    print(f"Electra Avg:        {df['electra_sim_out_vs_ramsay'].mean():.2f}")

Loading Judge: unsloth/Qwen2.5-7B-Instruct-bnb-4bit...
==((====))==  Unsloth 2026.2.1: Fast Qwen2 patching. Transformers: 4.56.2.
   \\   /|    NVIDIA L4. Num GPUs = 1. Max memory: 22.034 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.0+cu128. CUDA: 8.9. CUDA Toolkit: 12.8. Triton: 3.5.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.33.post1. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Running Qwen 2.5 7B Judge...


  0%|          | 0/100 [00:00<?, ?it/s]

Avg Qwen 2.5 7B Similarity: 1.98
Saved: /content/drive/MyDrive/RamsayData/Outcome/ramsay_judge_qwen25_7b.csv

--- Final Comparison of All Judges ---
Llama 3.1 8B Avg:   3.23
Qwen 2.5 1.5B Avg:  5.00
Qwen 2.5 7B Avg:    1.98
Electra Avg:        2.02
